# 05 Chart And Signal Scanner

Scanner dùng để nhìn tín hiệu trực quan trên chart. Đây không phải backtest đầy đủ:
scanner không mô phỏng lot sizing, commission, swap, pending fill như execution engine.

Mục tiêu notebook:
- Xem tín hiệu một symbol trên chart.
- Scan nhiều symbol để tìm symbol đang có setup đáng quan tâm.
- Tóm tắt pass/rejected/win/outcome theo cách dễ nhìn hơn.


In [ ]:
# Cell 1 - Bootstrap đường dẫn import an toàn
#
# Notebook có thể được mở từ repo root, từ thư mục strategies/combo,
# hoặc từ một working directory khác trong VS Code/Jupyter. Vì vậy ta không
# dùng `config.py` làm marker root: trong strategies/combo cũng có config.py,
# rất dễ nhận nhầm thư mục strategy là repo root.
#
# Marker đáng tin cậy hơn là pyproject.toml + thư mục core_python/shared.
# Sau khi tìm được root thật, ta thêm cả repo root và core_python vào sys.path
# để import được `shared.*` và `strategies.combo.*`.

import sys
from pathlib import Path


def _find_root(start: Path, marker: str = 'pyproject.toml') -> Path:
    for p in [start, *start.parents]:
        if (p / marker).exists() and (p / 'core_python' / 'shared').exists():
            return p
    raise RuntimeError(f'Không tìm thấy repo root chứa {marker!r} và core_python/shared')


ROOT = _find_root(Path.cwd())
CORE = ROOT / 'core_python'
for p in (str(ROOT), str(CORE)):
    if p not in sys.path:
        sys.path.insert(0, p)

print('ROOT =', ROOT)
print('CORE =', CORE)


In [ ]:
# Cell 2 - Import thư viện và helper scanner

import pandas as pd
from IPython.display import display

from strategies.combo.config import SYMBOLS, get_indicator_params, summary as strategy_summary
from strategies.combo.notebook_utils import (
    configure_notebook,
    plot_scan_summary,
    show_note,
    show_run_config,
)
from strategies.combo.scanner import (
    build_reversal_figure,
    calc_reversal_stats,
    run_multi_reversal_scan,
    run_reversal_scan,
)

configure_notebook()
print(strategy_summary())
print('Symbols:', ', '.join(SYMBOLS.keys()))


In [ ]:
# Cell 3 - Cấu hình scanner
#
# `n_bars`: số nến gần nhất để xem trên chart.
# `scan_params_overrides`: chỉnh nhanh MA/KTP/MIN_RR mà không sửa config.py.

RUN_CONFIG = {
    'scan_symbol': 'US30',
    'scan_symbols': ['US30', 'US500', 'DE40', 'GOLD', 'BTCUSD'],
    'n_bars': 250,
    'scan_params_overrides': {
        # 'MA_PERIOD': 20,
        # 'KTP': 2.3,
        # 'MIN_RR': 1.25,
    },
}

SCAN_PARAMS = get_indicator_params()
SCAN_PARAMS.update(RUN_CONFIG['scan_params_overrides'])
show_run_config('Cấu hình scanner', RUN_CONFIG)
show_note('SCAN_PARAMS hiệu lực', 'Đây là bộ tham số indicator/signal đang dùng cho scanner.')
display(SCAN_PARAMS)


In [ ]:
# Cell 4 - Scan một symbol
#
# Bảng signals_df là danh sách tín hiệu scanner phát hiện. Hãy đọc cùng chart ở cell sau,
# vì scanner chủ yếu phục vụ kiểm tra trực quan.

single = run_reversal_scan(RUN_CONFIG['scan_symbol'], RUN_CONFIG['n_bars'], SCAN_PARAMS)
single_stats = calc_reversal_stats(single['signals_df'])
show_note('Single-symbol scanner stats', 'Thống kê nhanh cho symbol đang xem.')
display(pd.DataFrame([single_stats]).T.rename(columns={0: 'value'}))
display(single['signals_df'].tail(30))


In [ ]:
# Cell 5 - Chart tín hiệu một symbol
#
# Chart này giúp kiểm tra tín hiệu có hợp lý về mặt cấu trúc giá không.
# Nếu tín hiệu nhìn vô lý trên chart, chưa nên tin kết quả backtest.

fig = build_reversal_figure(RUN_CONFIG['scan_symbol'], single, SCAN_PARAMS)
fig.show()


In [ ]:
# Cell 6 - Scan nhiều symbol và dashboard summary
#
# Dùng để tìm symbol hiện có nhiều setup đạt RR hoặc outcome tốt hơn.

multi = run_multi_reversal_scan(RUN_CONFIG['scan_symbols'], RUN_CONFIG['n_bars'], SCAN_PARAMS)
summary_rows = []
for sym, scan_result in multi.items():
    summary_rows.append({'symbol': sym, **calc_reversal_stats(scan_result['signals_df'])})

summary_df = pd.DataFrame(summary_rows)
if not summary_df.empty:
    sort_cols = [c for c in ['n_pass', 'win_pct', 'avg_rr'] if c in summary_df.columns]
    summary_df = summary_df.sort_values(sort_cols, ascending=False, ignore_index=True)
    display(summary_df)
    plot_scan_summary(summary_df)
else:
    print('Không có summary scanner.')


In [ ]:
# Cell 7 - Mở chart của candidate tốt nhất theo scanner
#
# Candidate này chỉ là gợi ý trực quan, không phải kết luận hiệu suất.
# Muốn đánh giá hiệu quả, hãy chạy 01_symbol_backtest.ipynb cho symbol đó.

selected = summary_df.iloc[0]['symbol'] if not summary_df.empty else RUN_CONFIG['scan_symbol']
print('Best current scanner candidate =', selected)
selected_result = multi.get(selected) or single
selected_fig = build_reversal_figure(selected, selected_result, SCAN_PARAMS)
selected_fig.show()
